In [ ]:
def read_lines_in_chunks(filename, chunk_size=1000):
    with open(filename, 'r') as file:
        while True:
            chunk = []
            for _ in range(chunk_size):
                line = file.readline()
                if not line:
                    break
                chunk.append(line.strip())
            if not chunk:
                break
            yield chunk
            break

# Initialize
from bcrm.paths import data_path
filename = data_path('foundation/diverse_foundation_data_protein.csv')
training_corpus = read_lines_in_chunks(filename)

import tqdm
from transformers import AutoTokenizer
print("initialized...")
old_tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

print("training...")
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 10)
#tokenizer.add_tokens(['[cls1]', '[cls2]', '[cls3]', '[cls4]'])

print("saving...")
tokenizer.save_pretrained("protein_bpe_tokenizer")  # committed at foundation-models/protein_bpe_tokenizer
tokenizer.get_vocab()


In [ ]:
tokenizer.vocab_size
#len(tokenizer.get_vocab())
#tokenizer.get_vocab()

In [ ]:
#from transformers import PreTrainedTokenizerFast
#tokenizer = PreTrainedTokenizerFast(tokenizer_file="path/to/corpus_bpe_tokenizer/tokenizer.json")

import numpy as np

def get_lengths(dictionary):
    # Get the lengths of all keys in the dictionary
    lengths = [len(key) for key in dictionary.keys()]
    
    # Calculate the median, minimum, and maximum lengths
    lengths.sort()  # Sort the lengths in ascending order
    n = len(lengths)
    
    # Calculate median
    median_length = lengths[n // 2] if n % 2 != 0 else (lengths[n // 2 - 1] + lengths[n // 2]) / 2
    
    # Minimum and maximum lengths
    min_length = min(lengths)
    max_length = max(lengths)
    
    # Calculate mean length
    mean_length = np.mean(lengths)
    
    return median_length, min_length, max_length, mean_length

# Get the lengths
median_length, min_length, max_length, mean_length = get_lengths(tokenizer.get_vocab())

# Print the results
print("Median Length:", median_length)
print("Minimum Length:", min_length)
print("Maximum Length:", max_length)
print("Mean:", mean_length)

In [ ]:
#tokenizer.get_vocab()
#tokenizer.add_tokens(['cls1', '[cls1]'])
tokenizer.get_vocab()

In [ ]:
#from tokenizers import Tokenizer

# Load trained tokenizer
#tokenizer_save_path = "path/to/bpe_tokenizer/tokenizer.json"
#tokenizer = Tokenizer.from_file(tokenizer_save_path)

# Tokenize text
text = "GGGTCCCTGAGACTCTCCTGTACAGCCTCTGGATTCACCTTTAGCAGCTATGCCATGAGCTGGGTCCACCAGGCTCCAGGGAAGGGGCTGGAGTGGGTCTCAGCTATTAGTGGTAGTGGTGGTAGCACATACTACGCAGACTCCGTGAAGGGCCGGTTCACCATCTCCAGAGACAATTCCAAGAACACGCTGTATCTGCAAATGAACAGCCTGAGAGCCGAGGACACGGCCGTATATTACTGTGCGAAAGACAGTGGGAGCTACCACGCTGATTTTGACTACTGGGGCCAGGGAACCCTGGTCACCGTCTCCTCAGGGAGTGCATCCGCCCCAACCCTTTTCCCCCTCGTCTGACATGAAT"
encoded = tokenizer.encode(text,)
print(encoded.ids)
print(encoded.attention_mask)
encoded.tokens

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from tokenizers import Tokenizer
import random

class CustomTextDataset(Dataset):
    def __init__(self, data_dir, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []

        # Iterate over files in the directory
        for filename in os.listdir(data_dir):
            class_label = os.path.splitext(filename)[0]  # Use file name as class label
            file_path = os.path.join(data_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as file:
                for line in file:
                    line = line.strip()  # Remove leading/trailing whitespace
                    if line:  # Skip empty lines
                        self.examples.append((class_label, line))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        class_label, text = self.examples[idx]
        text = "[<start>]" + text + "[<end>]"
        encoded = self.tokenizer.encode(text)
        ids = encoded.ids
        mask = encoded.attention_mask
        token_len = len(encoded.ids)
        
        if token_len < max_length:
            ids = [0] * (max_length - token_len) + ids
            mask = [0] * (max_length - token_len) + mask
            
        if token_len > max_length:
            ids = ids[:max_length]
            mask = mask[:max_length]

        ids = torch.tensor(ids)
        mask = torch.tensor(mask)
        
        return {
            'input_ids': ids,
            'attention_mask': mask,
            'class_label': class_label,
            'length': token_len,
        }

# Example usage
data_dir = "path/to/downsample_comb"  # exploratory; data not part of the release
tokenizer_save_path = "path/to/bpe_tokenizer/tokenizer.json"
tokenizer = Tokenizer.from_file(tokenizer_save_path)

max_length = 128
dataset = CustomTextDataset(data_dir, tokenizer, max_length)

batch_size = 256  # Adjust batch size as needed
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Example usage to print raw tokens for the first 5 rows
count = 0
tensor_lengths = []

for batch in data_loader:
    lengths = batch['length']

    for length in lengths:
    
        # Append length to list
        tensor_lengths.append(length)

        count += 1
        if count >= 50000:
            break
    if count >= 50000:
        break


In [ ]:
# Calculate average length
average_length = np.mean(tensor_lengths)

# Calculate 90th percentile length
percentile_90 = np.percentile(tensor_lengths, 99.9)

# Calculate 10th percentile length
percentile_10 = np.percentile(tensor_lengths, 0.1)

# Report results
print("Average Length:", average_length)
print("99th Percentile Length:", percentile_90)
print("1st Percentile Length:", percentile_10)